# LeetCode 599 — Minimum Index Sum of Two Lists bằng Python

**Link đề:** https://leetcode.com/problems/minimum-index-sum-of-two-lists/
**Độ khó:** Easy · **Chủ đề:** Array, Hash Table, String

```python
class Solution:
    def findRestaurant(self, list1: list[str], list2: list[str]) -> list[str]:
        ...
```

## Mục tiêu học
1. Hiểu đúng đề: tìm các chuỗi **chung** của hai list có **tổng chỉ số `i + j` nhỏ nhất**, trả về **tất cả** chuỗi hòa nhau.
2. Thành thạo `enumerate`, `dict` (chuỗi → chỉ số), dict comprehension, `set` và phép giao `&`.
3. Đi từ brute force hai vòng lặp `O(n1·n2)` → **hash map** một lượt `O(n1 + n2)`; biết thêm cách **duyệt theo tổng chỉ số** dừng sớm.
4. Nắm kỹ thuật "**giữ min và list các phần tử đạt min**" (reset khi có min mới, thêm vào khi bằng min).
5. Trace, chứng minh tính đúng, Time/Space, 6 bài tập với gợi ý và đáp án cuối.

## 1. Đề bài và ví dụ

Cho hai list chuỗi `list1`, `list2`. Một **chuỗi chung** là chuỗi xuất hiện ở cả hai. Nếu chuỗi đó nằm ở `list1[i]` và `list2[j]` thì **tổng chỉ số** là `i + j`.

Trả về **tất cả** chuỗi chung có tổng chỉ số **nhỏ nhất** (thứ tự tùy ý).

| list1 | list2 | Output |
|---|---|---|
| `["Shogun","Tapioca Express","Burger King","KFC"]` | `["Piatti","The Grill at Torrey Pines","Hungry Hunter Steakhouse","Shogun"]` | `["Shogun"]` |
| `["Shogun","Tapioca Express","Burger King","KFC"]` | `["KFC","Shogun","Burger King"]` | `["Shogun"]` |
| `["happy","sad","good"]` | `["sad","happy","good"]` | `["sad","happy"]` |

Ví dụ 2: chung có `Shogun` (0+1=1), `Burger King` (2+2=4), `KFC` (3+0=3) → nhỏ nhất là `Shogun`.
Ví dụ 3: `happy` (0+1=1), `sad` (1+0=1), `good` (2+2=4) → **hai** chuỗi hòa nhau ở tổng 1.

**Constraints:**
- `1 <= len(list1), len(list2) <= 1000`; mỗi chuỗi dài `1..30`, gồm chữ cái tiếng Anh và dấu cách.
- Các chuỗi **trong cùng một list là khác nhau** (unique).
- **Luôn có ít nhất một** chuỗi chung.

## 2. Nền tảng Python

### 2.1. `enumerate` – lấy cả chỉ số lẫn giá trị
`for i, s in enumerate(lst)` cho từng cặp `(chỉ số, phần tử)`. Gọn hơn `for i in range(len(lst)): s = lst[i]`.

In [1]:
list1 = ["happy", "sad", "good"]
for i, s in enumerate(list1):
    print(i, s)

0 happy
1 sad
2 good


### 2.2. `dict` chuỗi → chỉ số
Ta muốn hỏi nhanh "chuỗi `s` nằm ở chỉ số nào trong `list1`?". Lưu `index1[s] = i`. Tra `s in index1` trung bình `O(1)`, đọc `index1[s]` lấy chỉ số.

**Dict comprehension** `{s: i for i, s in enumerate(list1)}` tạo dict trong một dòng.

Vì đề bảo đảm các chuỗi trong list là **khác nhau**, mỗi key chỉ ứng với đúng một chỉ số – không lo bị ghi đè.

In [2]:
index1 = {s: i for i, s in enumerate(list1)}
print(index1)
print("'sad' in index1 ?", "sad" in index1, "-> chỉ số", index1["sad"])
print("'bad' in index1 ?", "bad" in index1)

{'happy': 0, 'sad': 1, 'good': 2}
'sad' in index1 ? True -> chỉ số 1
'bad' in index1 ? False


### 2.3. `set` và phép giao `&`
`set` là tập hợp không trùng, kiểm tra thành viên nhanh. `set(a) & set(b)` cho các phần tử **chung**. Lưu ý: set **không giữ thứ tự**.

In [3]:
list2 = ["sad", "happy", "good"]
common = set(list1) & set(list2)
print("Chuỗi chung:", sorted(common))     # sorted để in ổn định

Chuỗi chung: ['good', 'happy', 'sad']


### 2.4. Kỹ thuật "min + danh sách hòa"
Khi cần **mọi** phần tử đạt giá trị nhỏ nhất trong một lượt duyệt:
- gặp giá trị **nhỏ hơn** min → cập nhật min, **reset** list về `[phần tử]`;
- gặp giá trị **bằng** min → **thêm** vào list;
- lớn hơn → bỏ qua.

In [4]:
scores = [("An", 7), ("Bình", 5), ("Chi", 9), ("Dung", 5)]
best, names = float("inf"), []
for name, sc in scores:
    if sc < best:
        best, names = sc, [name]
    elif sc == best:
        names.append(name)
print("min =", best, "| đạt min:", names)

min = 5 | đạt min: ['Bình', 'Dung']


`float("inf")` là "vô cực dương" – nhỏ hơn nó là mọi số thực, rất tiện để khởi tạo min.

## 3. Cách 1 — Brute force: thử mọi cặp `(i, j)`

Với mỗi `i`, với mỗi `j`, nếu `list1[i] == list2[j]` thì đó là chuỗi chung với tổng `i + j`. Áp dụng kỹ thuật "min + danh sách hòa".

In [5]:
def find_restaurant_bruteforce(list1, list2):
    best = float("inf")
    result = []
    for i in range(len(list1)):
        for j in range(len(list2)):
            if list1[i] == list2[j]:
                s = i + j
                if s < best:
                    best, result = s, [list1[i]]
                elif s == best:
                    result.append(list1[i])
    return result


L1 = ["Shogun", "Tapioca Express", "Burger King", "KFC"]
L2a = ["Piatti", "The Grill at Torrey Pines", "Hungry Hunter Steakhouse", "Shogun"]
L2b = ["KFC", "Shogun", "Burger King"]
print(find_restaurant_bruteforce(L1, L2a))
print(find_restaurant_bruteforce(L1, L2b))
print(find_restaurant_bruteforce(["happy", "sad", "good"], ["sad", "happy", "good"]))

['Shogun']
['Shogun']
['happy', 'sad']


**Nút thắt:** `n1 · n2 = 10^6` lần so sánh chuỗi (mỗi lần tới 30 ký tự). Vẫn chạy được với constraints này, nhưng phần lặp lại là: với mỗi `list1[i]`, ta **quét lại cả `list2`** để hỏi "chuỗi này ở đâu trong list2?". Câu hỏi "ở đâu?" chính là việc của **hash map**.

## 4. Insight — Lưu vị trí một list vào dict, duyệt list còn lại

1. `index1 = {s: i for i, s in enumerate(list1)}` – mất `O(n1)`.
2. Duyệt `j, s` trong `list2`: nếu `s in index1` thì tổng là `index1[s] + j`.
3. Áp dụng "min + danh sách hòa".

Tính tay ví dụ 3: `index1 = {happy:0, sad:1, good:2}`.

| j | s | s in index1? | tổng | best | result |
|---:|---|:---:|---:|---:|---|
| 0 | sad | có (1) | 1 | 1 | `[sad]` |
| 1 | happy | có (0) | 1 | 1 | `[sad, happy]` |
| 2 | good | có (2) | 4 | 1 | `[sad, happy]` |

## 5. Cách 2 — Hash map một lượt: giảng từng đoạn rồi ghép

In [6]:
def find_restaurant_hashmap(list1, list2):
    index1 = {s: i for i, s in enumerate(list1)}   # B1: chuỗi -> chỉ số trong list1
    best = float("inf")
    result = []
    for j, s in enumerate(list2):                  # B2: duyệt list2
        if s in index1:                            # chuỗi chung
            total = index1[s] + j
            if total < best:                       # B3: min mới -> reset
                best, result = total, [s]
            elif total == best:                    #     hòa -> thêm
                result.append(s)
    return result


print(find_restaurant_hashmap(L1, L2a))
print(find_restaurant_hashmap(L1, L2b))
print(find_restaurant_hashmap(["happy", "sad", "good"], ["sad", "happy", "good"]))

['Shogun']
['Shogun']
['sad', 'happy']


### 5.1. TRACE cách hash map

In [7]:
def trace_hashmap(list1, list2):
    index1 = {s: i for i, s in enumerate(list1)}
    print("index1 =", index1)
    best, result = float("inf"), []
    for j, s in enumerate(list2):
        if s in index1:
            total = index1[s] + j
            if total < best:
                best, result = total, [s]
                act = "min mới -> reset"
            elif total == best:
                result.append(s)
                act = "hòa -> thêm"
            else:
                act = "lớn hơn -> bỏ"
            print(f"j={j} '{s}': i={index1[s]}, tổng={total} | {act:<16} | best={best}, result={result}")
        else:
            print(f"j={j} '{s}': không có trong list1")
    return result

trace_hashmap(L1, L2b)
print()
trace_hashmap(["happy", "sad", "good"], ["sad", "happy", "good"])

index1 = {'Shogun': 0, 'Tapioca Express': 1, 'Burger King': 2, 'KFC': 3}
j=0 'KFC': i=3, tổng=3 | min mới -> reset | best=3, result=['KFC']
j=1 'Shogun': i=0, tổng=1 | min mới -> reset | best=1, result=['Shogun']
j=2 'Burger King': i=2, tổng=4 | lớn hơn -> bỏ    | best=1, result=['Shogun']

index1 = {'happy': 0, 'sad': 1, 'good': 2}
j=0 'sad': i=1, tổng=1 | min mới -> reset | best=1, result=['sad']
j=1 'happy': i=0, tổng=1 | hòa -> thêm      | best=1, result=['sad', 'happy']
j=2 'good': i=2, tổng=4 | lớn hơn -> bỏ    | best=1, result=['sad', 'happy']


['sad', 'happy']

### 5.2. Tối ưu nhỏ: dừng sớm
Khi `j > best`, mọi tổng sau đó `index1[s] + j >= j > best` → không thể tốt hơn hay hòa → có thể `break`. Không đổi Big-O nhưng chạy nhanh hơn trong thực tế.

Thêm nữa: dựng dict từ list **ngắn hơn** để tiết kiệm bộ nhớ (kết quả không đổi vì `i + j` đối xứng).

In [8]:
def find_restaurant_early_stop(list1, list2):
    if len(list1) > len(list2):                  # dict cho list ngắn hơn
        list1, list2 = list2, list1
    index1 = {s: i for i, s in enumerate(list1)}
    best, result = float("inf"), []
    for j, s in enumerate(list2):
        if j > best:
            break                                # không thể tốt hơn nữa
        if s in index1:
            total = index1[s] + j
            if total < best:
                best, result = total, [s]
            elif total == best:
                result.append(s)
    return result

print(find_restaurant_early_stop(L1, L2b))
print(sorted(find_restaurant_early_stop(["happy", "sad", "good"], ["sad", "happy", "good"])))

['Shogun']
['happy', 'sad']


## 6. Cách 3 — Duyệt theo **tổng chỉ số** tăng dần

Ý tưởng khác hẳn: thử `total = 0, 1, 2, ...`. Với mỗi `total`, xét mọi cặp `(i, total − i)`; nếu `list1[i]` có trong `set(list2)` **và** đúng ở vị trí `total − i` thì là đáp án. Gặp `total` đầu tiên có ít nhất một chuỗi → trả về luôn (đây chắc chắn là tổng nhỏ nhất).

Cách này đẹp vì **dừng ngay khi tìm thấy**, rất nhanh nếu chuỗi chung nằm gần đầu. Tệ nhất `O((n1 + n2)²)`.

In [9]:
def find_restaurant_by_sum(list1, list2):
    index2 = {s: j for j, s in enumerate(list2)}
    n1, n2 = len(list1), len(list2)
    for total in range(n1 + n2 - 1):
        found = []
        for i in range(max(0, total - (n2 - 1)), min(total, n1 - 1) + 1):
            s = list1[i]
            if index2.get(s) == total - i:
                found.append(s)
        if found:
            return found
    return []

print(find_restaurant_by_sum(L1, L2a))
print(find_restaurant_by_sum(L1, L2b))
print(find_restaurant_by_sum(["happy", "sad", "good"], ["sad", "happy", "good"]))

['Shogun']
['Shogun']
['happy', 'sad']


Giải thích khoảng `i`: cần `0 <= i <= n1 − 1` và `0 <= j = total − i <= n2 − 1` ⇒ `total − (n2 − 1) <= i <= total`. Kết hợp lại: `i` chạy từ `max(0, total − n2 + 1)` tới `min(total, n1 − 1)`.

`dict.get(key)` trả `None` nếu không có key, nên `index2.get(s) == total - i` an toàn.

In [10]:
def trace_by_sum(list1, list2):
    index2 = {s: j for j, s in enumerate(list2)}
    n1, n2 = len(list1), len(list2)
    for total in range(n1 + n2 - 1):
        pairs = [(i, total - i) for i in range(max(0, total - n2 + 1), min(total, n1 - 1) + 1)]
        found = [list1[i] for i, j in pairs if index2.get(list1[i]) == j]
        print(f"total={total}: các cặp (i,j)={pairs} -> tìm thấy {found}")
        if found:
            return found

trace_by_sum(L1, L2b)

total=0: các cặp (i,j)=[(0, 0)] -> tìm thấy []
total=1: các cặp (i,j)=[(0, 1), (1, 0)] -> tìm thấy ['Shogun']


['Shogun']

## 7. Code nộp LeetCode

In [11]:
class Solution:
    def findRestaurant(self, list1: list[str], list2: list[str]) -> list[str]:
        index1 = {s: i for i, s in enumerate(list1)}
        best = float("inf")
        result = []
        for j, s in enumerate(list2):
            if j > best:
                break
            if s in index1:
                total = index1[s] + j
                if total < best:
                    best, result = total, [s]
                elif total == best:
                    result.append(s)
        return result


sol = Solution()
assert sol.findRestaurant(L1, L2a) == ["Shogun"]
assert sol.findRestaurant(L1, L2b) == ["Shogun"]
assert sorted(sol.findRestaurant(["happy", "sad", "good"], ["sad", "happy", "good"])) == ["happy", "sad"]
print("Solution PASS ví dụ đề")

Solution PASS ví dụ đề


Bản "Pythonic" ngắn (2 lượt): tính tổng cho mọi chuỗi chung rồi lấy những chuỗi có tổng bằng min.

In [12]:
class SolutionPythonic:
    def findRestaurant(self, list1: list[str], list2: list[str]) -> list[str]:
        index1 = {s: i for i, s in enumerate(list1)}
        sums = {s: index1[s] + j for j, s in enumerate(list2) if s in index1}
        best = min(sums.values())
        return [s for s, t in sums.items() if t == best]

assert sorted(SolutionPythonic().findRestaurant(["happy", "sad", "good"], ["sad", "happy", "good"])) == ["happy", "sad"]
print("SolutionPythonic PASS")

SolutionPythonic PASS


## 8. Vì sao đúng?

**Không sót:** mọi chuỗi chung `s` đều xuất hiện đúng một lần trong `list2` (do unique) tại chỉ số `j`, và `index1[s]` là **chỉ số duy nhất** của nó trong `list1`. Vì vậy khi duyệt `list2`, mỗi chuỗi chung được xét đúng một lần với đúng tổng `i + j`.

**Invariant của vòng lặp:** sau khi xét `list2[0..j]`, `best` là tổng nhỏ nhất trong các chuỗi chung đã gặp, `result` là **tất cả** chuỗi chung đã gặp có tổng bằng `best`.
- Gặp tổng nhỏ hơn: các phần tử cũ trong `result` không còn đạt min → reset đúng.
- Gặp tổng bằng: thêm vào → vẫn đủ.
- Gặp tổng lớn hơn: không thuộc kết quả → bỏ đúng.

**Dừng sớm đúng:** nếu `j > best` thì mọi chuỗi sau có tổng `>= j > best`.

**Cách 3 đúng:** duyệt `total` tăng dần nên `total` đầu tiên có chuỗi chung chính là tổng nhỏ nhất; ta liệt kê đủ mọi cặp `(i, total − i)` hợp lệ nên không sót chuỗi hòa.

## 9. Time / Space Complexity và so sánh

`n1, n2` = độ dài hai list, `L` = độ dài chuỗi tối đa (≤ 30). Hash/so sánh chuỗi tốn `O(L)`.

| Cách | Time | Space | Ghi chú |
|---|---|---|---|
| 1. Brute force | `O(n1 · n2 · L)` | `O(1)` ngoài output | Dễ nghĩ nhất |
| 2. Hash map | `O((n1 + n2) · L)` | `O(n1 · L)` | Bản chính |
| 2b. Hash map + dừng sớm, dict cho list ngắn | như trên | `O(min(n1, n2) · L)` | Nhanh hơn thực tế |
| 3. Duyệt theo tổng | `O((n1 + n2)² · L)` tệ nhất | `O(n2 · L)` | Dừng ngay khi gặp, hay khi đáp án ở đầu |
| Pythonic 2 lượt | `O((n1 + n2) · L)` | `O((n1 + n2) · L)` | Ngắn, dễ đọc |

Thường ta coi `L` là hằng số → cách 2 là `O(n1 + n2)`.

## 10. Bộ test: mẫu đề, biên, đối chiếu ngẫu nhiên

Vì thứ tự output tùy ý, ta so sánh bằng `sorted(...)`.

In [13]:
import random

methods = {
    "bruteforce": find_restaurant_bruteforce,
    "hashmap": find_restaurant_hashmap,
    "early_stop": find_restaurant_early_stop,
    "by_sum": find_restaurant_by_sum,
    "Solution": Solution().findRestaurant,
    "Pythonic": SolutionPythonic().findRestaurant,
}
cases = [
    ("đề 1", L1, L2a, ["Shogun"]),
    ("đề 2", L1, L2b, ["Shogun"]),
    ("đề 3 (hòa)", ["happy", "sad", "good"], ["sad", "happy", "good"], ["happy", "sad"]),
    ("1 phần tử", ["A"], ["A"], ["A"]),
    ("chung ở cuối", ["a", "b", "c"], ["x", "y", "c"], ["c"]),
    ("hòa 3 chuỗi", ["a", "b", "c"], ["c", "b", "a"], ["a", "b", "c"]),
    ("có dấu cách", ["Pho Ha Noi", "Bun Cha"], ["Bun Cha", "Pho Ha Noi"], ["Bun Cha", "Pho Ha Noi"]),
    ("phân biệt hoa thường", ["kfc", "KFC"], ["KFC"], ["KFC"]),
]
for name, a, b2, expected in cases:
    for mname, f in methods.items():
        got = sorted(f(a, b2))
        assert got == sorted(expected), (name, mname, got)
    print(f"{name:<22} PASS -> {sorted(expected)}")

đề 1                   PASS -> ['Shogun']
đề 2                   PASS -> ['Shogun']
đề 3 (hòa)             PASS -> ['happy', 'sad']
1 phần tử              PASS -> ['A']
chung ở cuối           PASS -> ['c']
hòa 3 chuỗi            PASS -> ['a', 'b', 'c']
có dấu cách            PASS -> ['Bun Cha', 'Pho Ha Noi']
phân biệt hoa thường   PASS -> ['KFC']


In [14]:
rng = random.Random(599)
pool = [chr(ord("a") + k) for k in range(8)]
for _ in range(400):
    a = rng.sample(pool, rng.randint(1, 8))
    b2 = rng.sample(pool, rng.randint(1, 8))
    if not set(a) & set(b2):
        b2.append(a[0])                     # đảm bảo có chuỗi chung như đề
    expected = sorted(find_restaurant_bruteforce(a, b2))
    for mname, f in methods.items():
        assert sorted(f(a, b2)) == expected, (a, b2, mname)
print("400 bộ ngẫu nhiên: 6 cách khớp nhau — PASS")

400 bộ ngẫu nhiên: 6 cách khớp nhau — PASS


## 11. Lỗi người mới hay gặp

### Lỗi 1: Chỉ trả về **một** chuỗi
Dùng `if total < best` mà quên nhánh `elif total == best` → ví dụ 3 chỉ ra `["sad"]`.

### Lỗi 2: Dùng `<=` rồi luôn reset
`if total <= best: best, result = total, [s]` → chuỗi hòa sau xóa chuỗi hòa trước.

### Lỗi 3: Trả về chỉ số hoặc tổng thay vì chuỗi
Đề cần **list chuỗi**.

### Lỗi 4: Dùng `list2.index(s)` trong vòng lặp
`list.index` quét tuyến tính `O(n2)` → lại thành `O(n1 · n2)` dù trông "gọn".

In [15]:
def wrong_only_one(list1, list2):
    index1 = {s: i for i, s in enumerate(list1)}
    best, result = float("inf"), []
    for j, s in enumerate(list2):
        if s in index1 and index1[s] + j < best:
            best, result = index1[s] + j, [s]
    return result

def wrong_le_reset(list1, list2):
    index1 = {s: i for i, s in enumerate(list1)}
    best, result = float("inf"), []
    for j, s in enumerate(list2):
        if s in index1 and index1[s] + j <= best:
            best, result = index1[s] + j, [s]
    return result

ex3 = (["happy", "sad", "good"], ["sad", "happy", "good"])
print("Quên nhánh hòa:", wrong_only_one(*ex3))
print("Dùng <= reset :", wrong_le_reset(*ex3))
print("Đúng          :", find_restaurant_hashmap(*ex3))

Quên nhánh hòa: ['sad']
Dùng <= reset : ['happy']
Đúng          : ['sad', 'happy']


`*ex3` "mở gói" tuple thành hai đối số riêng: `f(*ex3)` ≡ `f(ex3[0], ex3[1])`.

### Lỗi 5: Dùng `set` giao rồi quên tính chỉ số
`set(list1) & set(list2)` chỉ cho biết chuỗi chung, **không** cho tổng chỉ số – vẫn cần dict vị trí.

## 12. Mẹo nhận dạng pattern

- Hỏi "phần tử này **nằm ở đâu** trong list kia?" lặp đi lặp lại → **dict giá trị → chỉ số**.
- Cần **tất cả** phần tử đạt min/max trong một lượt → kỹ thuật **reset khi tốt hơn, thêm khi bằng**.
- Đề có "unique" → mỗi giá trị chỉ một chỉ số, dict không bị ghi đè; nếu không unique phải nghĩ lại (lưu chỉ số **đầu tiên** hay list chỉ số?).
- "Thứ tự tùy ý" → test bằng `sorted` hoặc `set`.

## 13. Biến thể

**Biến thể A — Có trùng lặp trong list:** lấy chỉ số **đầu tiên** của mỗi chuỗi (`setdefault`) vì nó cho tổng nhỏ nhất.

**Biến thể B — Ba list:** tổng `i + j + k` nhỏ nhất – dựng dict cho hai list, duyệt list thứ ba.

**Biến thể C — Trả về kèm tổng nhỏ nhất:** `(best, result)`.

`dict.setdefault(k, v)` chỉ gán `v` nếu `k` **chưa có** – rất hợp để giữ lần xuất hiện đầu tiên.

In [16]:
first = {}
for i, s in enumerate(["a", "b", "a", "c", "b"]):
    first.setdefault(s, i)
print("chỉ số đầu tiên:", first)       # {'a': 0, 'b': 1, 'c': 3}

chỉ số đầu tiên: {'a': 0, 'b': 1, 'c': 3}


## 14. Bài tập (TODO – không gọi khi chưa làm)

**Bài 1 — Tính tay.** `list1 = ["x","y","z","w"]`, `list2 = ["w","z","y"]`. Liệt kê chuỗi chung và tổng chỉ số; đáp án?

**Bài 2 — Đoán output.** `{s: i for i, s in enumerate(["a","b","a"])}` in gì? (Chú ý key trùng.)

**Bài 3 — Viết lại** `findRestaurant` bằng hash map một lượt.

**Bài 4 — Biến thể A:** list có thể chứa chuỗi trùng lặp; tổng chỉ số dùng **lần xuất hiện đầu tiên** trong mỗi list.

**Bài 5 — Biến thể B:** ba list `l1, l2, l3`; trả về các chuỗi chung của cả ba có `i + j + k` nhỏ nhất (thứ tự tùy ý, giả sử có ít nhất một).

**Bài 6 — Mini challenge:** trả về **tất cả** chuỗi chung, sắp xếp theo tổng chỉ số tăng dần, hòa thì theo thứ tự chữ cái.

In [17]:
answer_ex1 = None    # TODO
answer_ex2 = None    # TODO


def find_restaurant_exercise(list1, list2):
    # TODO 1: dict chuỗi -> chỉ số của list1
    # TODO 2: duyệt list2 bằng enumerate
    # TODO 3: reset khi nhỏ hơn, thêm khi bằng
    pass


def find_restaurant_with_duplicates(list1, list2):
    # TODO: dùng setdefault để lấy chỉ số đầu tiên ở CẢ HAI list
    pass

# Kiểm tra sau khi làm:
# assert sorted(find_restaurant_exercise(["happy", "sad", "good"], ["sad", "happy", "good"])) == ["happy", "sad"]
# assert find_restaurant_with_duplicates(["a", "b", "a"], ["b", "a", "b"]) == ["a", "b"]  (thứ tự tùy ý)

In [18]:
def find_three_lists(l1, l2, l3):
    # TODO: dict cho l1, l2; duyệt l3
    pass


def all_common_sorted(list1, list2):
    # TODO: tạo list (tổng, chuỗi) rồi sắp xếp theo key
    pass

# Kiểm tra sau khi làm:
# assert sorted(find_three_lists(["a", "b", "c"], ["c", "a", "b"], ["b", "c", "a"])) == ["a", "b", "c"]
# assert all_common_sorted(["happy", "sad", "good"], ["sad", "happy", "good"]) == ["happy", "sad", "good"]

## 15. Gợi ý 3 mức

**Bài 1:** (1) Tìm chỉ số của từng chuỗi chung trong cả hai list. (2) `y`: 1+2, `z`: 2+1, `w`: 3+0. (3) Cả ba đều tổng 3.

**Bài 2:** (1) Dict comprehension gán lần lượt. (2) Key trùng thì **giá trị sau ghi đè** giá trị trước. (3) `'a'` cuối cùng ứng với chỉ số 2.

**Bài 3:** (1) Hash map + "min và danh sách hòa". (2) `enumerate`, `in`, `float('inf')`. (3) Xem mục 5.

**Bài 4:** (1) Chỉ số đầu tiên luôn cho tổng nhỏ nhất. (2) `first2.setdefault(s, j)`. (3) Duyệt `first2.items()` thay vì `enumerate(list2)`; một chuỗi chỉ được thêm một lần.

**Bài 5:** (1) Giống bài 3 nhưng kiểm tra `s in d1 and s in d2`. (2) Tổng `d1[s] + d2[s] + k`. (3) Reset/thêm như cũ.

**Bài 6:** (1) Thu thập `(tổng, chuỗi)` của mọi chuỗi chung. (2) `sorted(pairs)` tự so sánh tổng trước, chuỗi sau. (3) `[s for _, s in sorted(pairs)]`.

## 16. Đáp án tham khảo (chạy kiểm tra)

In [19]:
ans_ex1 = sorted(find_restaurant_hashmap(["x", "y", "z", "w"], ["w", "z", "y"]))   # ['w', 'y', 'z'] – cùng tổng 3
ans_ex2 = {s: i for i, s in enumerate(["a", "b", "a"])}                            # {'a': 2, 'b': 1}


def ans_find_restaurant(list1, list2):
    index1 = {s: i for i, s in enumerate(list1)}
    best, result = float("inf"), []
    for j, s in enumerate(list2):
        if s in index1:
            total = index1[s] + j
            if total < best:
                best, result = total, [s]
            elif total == best:
                result.append(s)
    return result


def ans_find_restaurant_with_duplicates(list1, list2):
    first1, first2 = {}, {}
    for i, s in enumerate(list1):
        first1.setdefault(s, i)
    for j, s in enumerate(list2):
        first2.setdefault(s, j)
    best, result = float("inf"), []
    for s, j in first2.items():
        if s in first1:
            total = first1[s] + j
            if total < best:
                best, result = total, [s]
            elif total == best:
                result.append(s)
    return result


def ans_find_three_lists(l1, l2, l3):
    d1 = {s: i for i, s in enumerate(l1)}
    d2 = {s: j for j, s in enumerate(l2)}
    best, result = float("inf"), []
    for k, s in enumerate(l3):
        if s in d1 and s in d2:
            total = d1[s] + d2[s] + k
            if total < best:
                best, result = total, [s]
            elif total == best:
                result.append(s)
    return result


def ans_all_common_sorted(list1, list2):
    index1 = {s: i for i, s in enumerate(list1)}
    pairs = [(index1[s] + j, s) for j, s in enumerate(list2) if s in index1]
    return [s for _, s in sorted(pairs)]


print("Bài 1:", ans_ex1, "| Bài 2:", ans_ex2)
assert ans_ex1 == ["w", "y", "z"]
assert ans_ex2 == {"a": 2, "b": 1}
assert sorted(ans_find_restaurant(["happy", "sad", "good"], ["sad", "happy", "good"])) == ["happy", "sad"]
assert sorted(ans_find_restaurant_with_duplicates(["a", "b", "a"], ["b", "a", "b"])) == ["a", "b"]
assert sorted(ans_find_restaurant_with_duplicates(["a", "x", "b"], ["b", "b", "a"])) == ["a", "b"]
assert sorted(ans_find_three_lists(["a", "b", "c"], ["c", "a", "b"], ["b", "c", "a"])) == ["a", "b", "c"]
assert ans_find_three_lists(["a", "b"], ["b", "a"], ["a"]) == ["a"]
assert ans_all_common_sorted(["happy", "sad", "good"], ["sad", "happy", "good"]) == ["happy", "sad", "good"]
print("Đáp án: tất cả test PASS")

Bài 1: ['w', 'y', 'z'] | Bài 2: {'a': 2, 'b': 1}
Đáp án: tất cả test PASS


## 17. Template nhớ nhanh và checklist submit

```python
class Solution:
    def findRestaurant(self, list1: list[str], list2: list[str]) -> list[str]:
        index1 = {s: i for i, s in enumerate(list1)}    # chuỗi -> vị trí
        best, result = float("inf"), []
        for j, s in enumerate(list2):
            if j > best:
                break                                   # dừng sớm (tùy chọn)
            if s in index1:
                total = index1[s] + j
                if total < best:
                    best, result = total, [s]           # tốt hơn -> reset
                elif total == best:
                    result.append(s)                    # hòa -> thêm
        return result
```

Câu nhớ: **"dict vị trí + reset khi nhỏ hơn, thêm khi bằng"**.

### Checklist
- [ ] Trả về **list chuỗi**, không phải chỉ số/tổng?
- [ ] Có nhánh `elif total == best` để giữ mọi chuỗi hòa?
- [ ] Không dùng `list.index` trong vòng lặp (tránh `O(n²)`)?
- [ ] Khởi tạo `best = float("inf")`?
- [ ] Nếu dừng sớm: điều kiện là `j > best` (không phải `>=`, kẻo bỏ sót chuỗi hòa có `i = 0`)?